# コーディングエージェントと開発の検証

テストを仕様として使い、結果を観察して修正する開発の流れを扱います。

# Karpathy Guidelines

以下の記事は、LLMにコードを書かせるときの悪い癖を抑えるための **「Karpathy Guidelines」** を紹介・解説したものです。中心は、Claude Codeなどで使える `SKILL.md` として、LLMのコーディング時の振る舞いを制御する考え方です。以下にこれを説明します。

Ref. https://zenn.dev/kotai/articles/050f0220c3f4de



## 要点

LLMはコード生成で便利だが、次のような問題を起こしやすい。

* 頼んでいない機能を追加する
* 関係ない箇所まで改善する
* 曖昧な指示を勝手に解釈する
* 本来短く書ける処理を過剰に複雑化する

これを防ぐために、記事では **4つの原則** を示している。



## 1. Think Before Coding

コードを書く前に、前提を確認せよ、という原則。

LLMは不明点があっても、黙ってもっともらしい解釈を選びがちである。
そのため、実装前に、

* 前提を明示する
* 複数解釈がある場合は提示する
* 不明点は質問する
* より単純な方法があれば提案する

ことが重要になる。



## 2. Simplicity First

最小限のコードで解決せよ、という原則。

LLMは「親切心」で、不要な抽象化、設定項目、将来拡張、過剰なエラーハンドリングを入れやすい。
しかし、それによりコードが肥大化し、理解・保守・レビューが難しくなる。

つまり、YAGNIの徹底である。

> 依頼された機能だけを、最も単純な形で実装する。


### YAGNIとは？
**YAGNI** は、

> **You Aren’t Gonna Need It**
> 「それ、たぶん今は必要ない」

というソフトウェア開発の原則です。

意味は、

> **将来必要になるかもしれない、という理由だけで機能・抽象化・設定項目を追加しない**

ということです。

例えば、ユーザーが

> JSONファイルを読んで辞書に変換する関数を書いて

と依頼した場合、YAGNI的には次で十分です。

```python
import json

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
```

一方、YAGNIに反する例は次のようなものです。

```python
class FileLoader:
    def load(self, path):
        raise NotImplementedError

class JsonLoader(FileLoader):
    def load(self, path):
        ...
        
class YamlLoader(FileLoader):
    def load(self, path):
        ...

class LoaderFactory:
    ...
```

将来 YAML も読むかもしれない、プラグイン化するかもしれない、複数形式に対応するかもしれない、という理由で最初から大きな設計にしている。
しかし、今の要求が JSON だけなら、それは過剰設計です。

LLMにコードを書かせる文脈では、YAGNIは特に重要です。LLMは「親切そうな拡張」を勝手に追加しやすいからです。





## 3. Surgical Changes

既存コードを編集するときは、必要な箇所だけを外科的に変更せよ、という原則。

これは、ユーザーが以前聞いていた「一部の更新を依頼したのに、全部書き直すことを防ぐ指示」に最も近い部分です。

LLMは、ついでにコメント、変数名、フォーマット、隣接コードを直したがる。
しかし、それをすると差分が大きくなり、レビューが困難になる。

したがって、

* 関係ないコードを改善しない
* 壊れていないものをリファクタリングしない
* 既存スタイルに合わせる
* 自分の変更で不要になったものだけ削除する
* 元からあるデッドコードは、頼まれない限り削除しない

という方針を取る。

#### リファクタリングとは、

プログラムの外から見た動作は変えずに、内部の書き方・構造を改善すること
です。機能追加やバグ修正とは少し違います。



## 4. Goal-Driven Execution

曖昧な依頼を、検証可能なゴールに変換せよ、という原則。

例えば、

* 「バグを直して」
  → バグを再現するテストを書き、それを通す
* 「バリデーションを追加して」
  → 不正入力のテストを書き、それを通す
* 「リファクタリングして」
  → 変更前後でテストが通ることを確認する

という形にする。

LLMに自律的に修正・検証を回させるには、「何をもって成功とするか」を明確にする必要がある。



## SKILL.mdとしての意味

この記事の重要点は、これらの原則を単なる心得ではなく、`SKILL.md` としてLLMに読み込ませる点にある。

つまり、毎回プロンプトで細かく指示するのではなく、

* 推測で進まない
* 余計な機能を足さない
* 変更範囲を最小にする
* テスト可能なゴールにする

という行動規範を、Claude Codeなどの開発環境に常駐させる設計である。


---



# Test-Driven Development



## TDD とは

TDD（テスト駆動開発）とは、

> 「先にテストを書くことで、仕様を明確化しながら開発する手法」

です。

通常の開発では：

```text
実装 → テスト
```

ですが、TDDでは逆に：

```text
テスト → 実装
```

の順で進めます。

---



## TDD の基本サイクル

有名なのは：

```text
Red → Green → Refactor
```

です。

### 1. Red

失敗するテストを書く。

例：

```python id="tq9kdc"
assert add(1, 2) == 3
```

まだ `add()` は存在しないので失敗する。

---

### 2. Green

最小限のコードを書いてテストを通す。

```python id="d8h2go"
def add(a, b):
    return a + b
```

---

### 3. Refactor

コードを整理する。

* 重複除去
* 可読性改善
* 構造整理

を行う。

ただし：

> テストは壊してはいけない

つまりテストが「安全網」になります。

---



## TDD の本質

TDDの重要点は、

> テストが「後から確認するもの」ではなく、
> 「仕様そのもの」になる

ことです。

例えば：

```python id="pp5m5q"
assert normalize("  A b ") == "a b"
```

というテストは、

* trimする
* lowercase化する
* 空白を整理する

という仕様を含んでいます。

つまり：

```text
テスト = 実行可能な仕様
```

です。

---



## TDD の利点

### 1. 仕様が明確になる

自然文だけでは曖昧。

しかしテストは具体的。

---

### 2. 回帰防止

後で修正しても、

```bash
pytest
```

で既存機能が壊れていないか確認できる。

---

### 3. リファクタリングしやすい

内部実装を大胆に変更できる。

---

### 4. 小さく実装する癖がつく

TDDでは：

```text
最小コードでまず通す
```

を重視する。

---



## 1. TDD と LLM agent の自動化ループ

### 古典的 TDD

従来の TDD では、人間が：

```text id="8ppsr8"
1. テストを書く
2. 実装する
3. テスト実行
4. 修正する
```

を行っていた。



### LLM agent による自動化

LLM agent 時代では、このループが自動化される。

agent は：

```text id="5vr4zf"
1. テストを読む
2. コード生成
3. pytest実行
4. failure log取得
5. 修正
6. 再実行
```

を繰り返す。


### execution-guided generation

これは：

```text id="bmb8w0"
生成
→ 実行
→ feedback
→ 再生成
```

という構造を持つ。

テスト結果や実行エラーを利用してコード生成を修正するため、

> execution-guided generation

と呼ばれる。



### self-correcting agent

agent は：

* SyntaxError
* test failure
* lint error
* type error

などを観測し、自律的に修正する。

つまり：

```text id="q0db6x"
失敗
→ 修正
→ 再試行
```

の自己修正ループを持つ。



### tool-using coding agent

agent は外部ツールも利用する。

代表例：

* pytest
* mypy
* flake8
* compiler
* git

など。

特に：

```text id="lq3g36"
pytest
```

は TDD-agent の中心的ツールである。



### Agent 時代の TDD の役割

TDD は agent に対して：

* 仕様制約
* 品質保証
* 回帰防止
* repository整合性維持

を与える。

特に：

```text id="m79v7m"
「既存テストを壊してはいけない」
```

という制約が重要である。





## 2. Coding agent は探索問題

### 通常のLLM

通常のLLMは：

```text id="8d4gbv"
P(next token | context)
```

を最大化する。

つまり：

> 「もっとも尤もらしい次トークン」

を生成するだけである。




## Coding agent

一方 coding agent は：

```text id="0uy1nx"
コード生成
→ 実行
→ エラー観測
→ 修正
→ 再実行
```

を繰り返す。

そのため本質的には：

> 「テストを満たすコード空間」

を探索している。





## 制約付き探索

通常のLLM：

```
maximize P(code | prompt)
```

coding agent：

```
maximize P(code | prompt)
subject to 
tests(code) = pass
```

つまり：

> 「テスト制約付き探索」

になっている。





## Trial-and-error search

coding agent は：

* compiler error
* pytest failure
* lint warning
* runtime error

などを利用して試行錯誤を行う。

そのため本質的には：

```text id="8mxf4j"
trial-and-error search
```

である。




## 3. Static generation から Interactive search へ

### 普通のLLM

通常のLLMは：

```text id="vf73m8"
入力→ 出力
```

の1回限りの生成を行う。

本質的には：

```text id="z17r2q"
1回の next-token prediction
```

である。

これは：

```text id="6c8b8n"
static generation
```

とみなせる。


### Agent化

Agent化すると、

```text id="rnq1h2"
状態
→ 行動
→ 実行
→ 観測
→ 次行動
```

という反復ループになる。



### 状態遷移系としての agent

agent は：

* 現在のコード
* テスト結果
* エラーログ
* repository状態

などを「状態」として保持する。

その状態に基づいて：

* コード編集
* テスト実行
* git操作
* lint実行

などの「行動」を選択する。



### Interactive search

この構造では、

```text id="ax7m6m"
文章生成
```

よりも、

```text id="7v6b3x"
探索
```

が中心になる。

つまり：

```text id="y5i0o3"
static generation
↓
interactive search
```

への変化が起きている。



### 本質的変化

従来のLLM：

```text id="1s0jzc"
「もっとも尤もらしい文章」を出力
```

Agent：

```text id="j5xwxt"
「目的を満たす状態」を探索
```

である。

そのため coding agent は、

単なる文章生成器ではなく、

> 「外部環境を利用する反復探索システム」

へ変化している。



---



## Agent における TDD の役割

### 1. 仕様固定

agent の暴走防止。

---

### 2. 自動評価

LLM 自身では正誤判定が弱い。

実行テストが external verifier になる。

---

### 3. マルチエージェント整合性

複数agentがいても：

```text
共通test suite
```

が契約になる。

---

### 4. repository-level stability

巨大コードベースで重要。

変更時に：

```bash
pytest
```

を回すことで整合性維持。

---



## TDDの代表的問題

> **hard coding（テストへの過剰適合）**

です。

例えばテストが：

```python id="9vqpm4"
assert square(2) == 4
assert square(3) == 9
```

だけの場合、LLM やプログラム生成器が：

```python id="xb7d18"
def square(x):
    if x == 2:
        return 4
    if x == 3:
        return 9
```

のように、

> 「テストだけ通るコード」

を書いてしまうことがあります。

これは：

```text id="ajijum"
本当の一般化
```

ではなく、

```text id="lym7kq"
テストケースへの暗記
```

です。

LLM agent では特に、

* テスト数が少ない
* hidden test がない
* specification が弱い

場合に起きやすいです。

そのため実務では：

* 境界ケースを増やす
* property-based testing を使う
* hidden test を使う
* fuzzing を行う

などで対策します。


# Agentic Harness Engineering (AHE) と Observability



## 1. 「LLMに最適なプロンプト」をLLM自身に聞いても不完全である

多くの人は：

```text id="6v5b9n"
「GPT-5.5に対する最適な質問方法」
```

を GPT-5.5 自身に尋ねる。

しかしここには本質的問題がある。

LLMは通常：

```text id="u0k1de"
過去世代のモデル出力
+
人間フィードバック
```

を含むデータで学習される。

つまり GPT-5.5 は：

```text id="k5m3ez"
GPT-4系以前の
「良いプロンプト文化」
```

を大量に学習している。

しかし：

```text id="2o7eaq"
現在のモデルで本当に最適な使い方
```

は、

```text id="q3m8ud"
現在モデル自身の能力
```

によって決まる。

したがって：

```text id="v2t6yl"
学習された「過去の最適」
≠
現在モデルでの真の最適
```

となる。

**つまり、GPT 5.5自身は自分に最適なプロンプトを知らない。**

これは：

```text id="m7f4ax"
モデル能力が変わる
↓
最適promptも変化する
```

ためである。

例えば旧世代LLMでは：

```text id="l3c7kn"
step-by-step を細かく指示
```

することが有効だった。

しかし高性能化したLLMでは：

```text id="r6h1pm"
細かすぎる手順固定
↓
探索能力を阻害
```

する場合がある。

つまり：

```text id="n0w9eu"
Prompt Engineering
```

自体が、

```text id="h7v2gr"
モデル進化と共に変化する対象
```

なのである。

この問題は、近年のAgent研究でさらに重要になった。

Agentでは：

```text id="y2r8sd"
prompt
```

だけでは性能が決まらない。

重要なのは：

```text id="j4l0bc"
LLMを囲む実行環境
```

である。

これを：

```text id="p9x6ta"
Harness
```

と呼ぶと以下の論文では定義している。

AHE論文：

〔Lin2026AHE〕
*Agentic Harness Engineering: Observability-Driven Automatic Evolution of Coding-Agent Harnesses*

は：

```text id="e4v1zn"
「prompt改善」
↓
「harness進化」
```

への転換を示した研究である。



## 2. Harnessとは何か？

Harnessとは：

> LLM Agent を動作させるための、
> モデル外部の制御・実行システム

である（と、この論文では定義している）。

AHE論文では：

* system prompt
* tools
* middleware
* memory
* sub-agents

などを含めて Harness と定義している〔Lin2026AHE〕。

通常のLLMは：

```text id="f0l4ae"
入力
→ 次token予測
→ 出力
```

のみを行う。

数理的には：
$$
P(x_t \mid x_{<t})
$$
を最大化するモデルである。

一方、Agentでは：

```text id="b8k2zn"
状態
→ 行動
→ 実行
→ 観測
→ 次行動
```

のループが発生する。

つまり：

```text id="t9m3ur"
static generation
↓
interactive execution
```

への変化である。



## ハーネスのソフトウェア例

| ソフトウェア | ハーネスとしての機能 |
|---|---|
| **Codex CLI（OpenAI）** | ターミナルで使うコーディングエージェントです。ファイル編集やコマンド実行を取り次ぎ、サンドボックスと承認設定で操作範囲を管理します。 [公式資料](https://learn.chatgpt.com/docs/agent-approvals-security) |
| **Claude Code（Anthropic）** | コーディングエージェントです。ツールごとに実行の許可・拒否や、実行前に確認を求める条件を設定できます。 [公式資料](https://code.claude.com/docs/en/permissions) |
| **OpenHands Software Agent SDK** | コーディングエージェントを構築するための SDK です。ファイル編集、コマンド実行、作業環境、ツールの追加を組み合わせられます。 [公式資料](https://docs.openhands.dev/sdk) |
| **LangChain Deep Agents** | 汎用エージェントの構築に使うハーネスです。タスク計画、仮想ファイルシステムによる作業情報の管理、サブエージェントへの委任を提供します。 [公式資料](https://www.langchain.com/deep-agents) |
| **Hermes Agent（Nous Research）** | CLI やデスクトップから使える汎用エージェントです。ツールの呼び出し、セッションをまたぐ記憶、作業手順を再利用するスキル、ローカルや Docker などの実行環境を管理します。 [公式資料](https://hermes-agent.nousresearch.com/ai-agent-platform) · [ソースコード](https://github.com/NousResearch/hermes-agent) |
| **TrueForge（TrueFoundry）** | モデルの呼び出し、MCP ツール、スキル、サンドボックス、承認、セッション状態を管理するエージェント実行基盤です。チャット UI から使うほか、HTTP API や TypeScript SDK を通じてアプリケーションにも組み込めます。 [公式ドキュメント](https://github.com/truefoundry/trueforge/blob/main/docs/introduction.mdx) · [ソースコード](https://github.com/truefoundry/trueforge) |

これらのソフトウェアは、モデルに作業を依頼してから、ツールの実行結果をモデルに返すまでの流れを管理します。たとえば文書の整理を依頼した場合は、モデルが必要なファイルを選び、ハーネスが許可された実行環境で読み取りツールを呼び出し、結果をモデルに返します。

### LangChain・LangGraph・Deep Agents の役割

| ソフトウェア | 主な役割 | RAG での例 |
|---|---|---|
| **LangChain** | LLM、埋め込み、検索、ツールを組み合わせるための部品を提供します。`create_agent` を使えば、モデルとツールから基本的なエージェントも作れます。 [公式資料](https://docs.langchain.com/oss/python/langchain/overview) | 質問を埋め込みに変換し、Chroma で関連文書を検索し、検索結果を LLM に渡す。 |
| **LangGraph** | 処理をノード、処理間の遷移をエッジとして定義する実行基盤です。共有状態、条件分岐、繰り返し、チェックポイントによる中断・再開を扱えます。 [公式資料](https://docs.langchain.com/oss/python/langgraph/overview) | 検索結果が不足した場合だけ質問を書き換えて再検索し、根拠が見つかれば回答へ進む。 |
| **Deep Agents** | LangChain のエージェントを土台に、計画、仮想ファイルシステムによる作業情報の管理、サブエージェントへの委任などを備えたハーネスです。 [公式資料](https://www.langchain.com/deep-agents) | 文献調査を複数の作業に分け、調査担当に検索を委任し、作業メモを使って結果を統合する。 |

LangChain の `create_agent` は LangGraph を実行基盤として使い、Deep Agents はその LangChain エージェントに機能を加えます。LangGraph は単独でも使えます。LangGraph でいうグラフは**処理の流れ**を表し、[Graph RAG](8300.0160.graph_RAG.ipynb) でいう知識グラフは**検索対象となる事実や関係**を表します。

このチュートリアルの [PDF 登録](8300.1101.RAG.build_chroma_db.ipynb) と [質問応答](8300.1201.RAG.ask_rag.ipynb) は、LangChain 系の部品を Python で順に呼び出す構成です。

### LangChain の典型例：プロンプトから回答を生成する

LangChain は、プロンプトの作成、モデルの呼び出し、出力の変換などを組み合わせるためのライブラリです。次の例では、文書と質問をテンプレートに埋め込み、モデルに渡し、返ってきた応答から文字列を取り出します。演算子 `|` でつないだ処理は左から順に実行されます。RAG に組み込む場合は、検索で得た文書を `context` に渡します。

次のコードを実行するには `langchain-openai` と `OPENAI_API_KEY` が必要です。

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "資料だけを根拠に答えてください。根拠がなければ不明と答えてください。"),
    ("human", "資料：{context}\n質問：{question}"),
])
model = ChatOpenAI(model="gpt-4.1-mini")
answer_chain = prompt | model | StrOutputParser()

answer = answer_chain.invoke({
    "context": "試料 S1 の ZnO 薄膜は、200 ℃で ALD 成膜した。",
    "question": "試料 S1 の成膜温度は何度ですか？",
})
print(answer)
```

資料から読み取れる回答は「200 ℃」です。 [LangChain 公式資料](https://docs.langchain.com/oss/python/langchain/overview)

### LangGraph の典型例：LLM の判定で次の処理を選ぶ

LangGraph では、検索する処理、LLM に聞く処理、回答を作る処理をそれぞれノードとして定義できます。条件付きエッジは、ノードの処理結果を見て次へ進む先を選びます。質問、検索語、取得文書、判定結果、検索回数などは共有状態に置きます。

たとえば、文献 Q&A では、検索結果と質問を LLM に渡し、「本文だけで回答できるか」を `sufficient` または `insufficient` のいずれかで返させます。プログラムはその値と検索回数を使って次のノードを選びます。

```text
質問 → 文書を検索 → LLM が根拠の十分さを判定
                     ├─ 十分 → LLM が回答を作る → 終了
                     ├─ 不足・まだ再検索していない → 検索語を書き換える → 文書を検索へ戻る
                     └─ 不足・再検索済み → 根拠不足を伝える → 終了
```

プロンプトと応答は各ノードの中で扱います。グラフに記述するのは、応答を受けて次にどの処理を実行するかです。分岐に加えて、前のノードに戻るループも表現できます。次の実行例では、検索結果の有無を Python で判定し、API キーなしで条件付きエッジを確認できます。 [LangGraph 公式資料](https://docs.langchain.com/oss/python/langgraph/overview)

### LangGraph の実行例：見つからなければ検索語を変える

短い文書集を検索します。初回で見つかれば終了し、見つからなければ検索語を変更して一度だけ再検索します。ノードは処理を実行し、条件付きエッジは共有状態を読んで次の処理を選びます。

実行するカーネルに `langgraph` が未導入なら、`%pip install langgraph` で導入します。この例は検索と分岐に焦点を当て、外部 API や LLM を使わず、文書のキーワード一致で動きます。

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class SearchState(TypedDict):
    query: str
    documents: list[str]
    attempts: int


documents = [
    "原子層堆積は、表面反応を順に繰り返す成膜法です。",
    "スパッタリングは、ターゲットから放出された粒子を基板に堆積させます。",
]
synonyms = {"ALD": "原子層堆積"}


def retrieve(state: SearchState) -> SearchState:
    query = state["query"]
    return {
        "query": query,
        "documents": [document for document in documents if query in document],
        "attempts": state["attempts"] + 1,
    }


def route_after_search(state: SearchState) -> Literal["rewrite_query", "__end__"]:
    if not state["documents"] and state["attempts"] == 1 and state["query"] in synonyms:
        return "rewrite_query"
    return END


def rewrite_query(state: SearchState) -> SearchState:
    return {**state, "query": synonyms[state["query"]]}


workflow = StateGraph(SearchState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_edge(START, "retrieve")
workflow.add_conditional_edges("retrieve", route_after_search)
workflow.add_edge("rewrite_query", "retrieve")
search_workflow = workflow.compile()

In [ ]:
for query in ("原子層堆積", "ALD", "熱蒸着"):
    initial_state = {"query": query, "documents": [], "attempts": 0}
    result = search_workflow.invoke(initial_state)
    print(
        f"入力: {query} | 最後の検索語: {result['query']} | "
        f"検索回数: {result['attempts']} | ヒット: {len(result['documents'])}"
    )

「原子層堆積」は1回で見つかり、「ALD」は検索語を置き換えて2回目で見つかります。「熱蒸着」は該当文書も置換候補もないため、1回で終了します。実際の RAG では、ここで取得した文書を回答を作るノードに渡せます。 [LangGraph 公式資料](https://docs.langchain.com/oss/python/langgraph/overview)

## 3. Harnessの構成要素



## System Prompt

Agentの行動方針。

例：

```text id="o2w8de"
- minimal edit
- test first
```

。



## Tool

Agentが利用可能な機能。

例：

```text id="x5k1ma"
run_shell_command
edit_file
git
```

。



## Middleware

Agent loop に介入する制御層。

例：

```text id="g9h6vp"
before_model()
after_tool()
```

。



## Long-term Memory

過去失敗や成功パターンを保存。

### Sub-agent

専門化された子Agent。



## 4. なぜHarnessが重要か？

AHEでは：

```text id="r4b8ln"
同じ GPT-5.4
```

を使いながら、

```text id="w1n6jk"
pass@1:
69.7%
→
77.0%
```

へ改善した〔Lin2026AHE〕。

つまり：

```text id="f5c2zs"
Agent性能
=
Model
+
Harness
```

である。



## 5. Observabilityとは何か？

AHEの中心概念が：

```text id="e8d7xa"
Observability
```

である。

論文の主張：

> 問題はAgent能力不足ではなく、
> Observability不足である。

Observabilityとは：

> Agent内部で
> 「なぜ失敗したか」
> 「どの変更が効いたか」
> を観測可能にすること

である。

| 制御理論     | AHE          |
| -------- | ------------ |
| 状態 ($x_t$) | harness状態    |
| 制御 ($u_t$) | edits        |
| 観測 ($y_t$) | trajectories |

として

制御理論では：

状態：
$$
x_{t+1} = f(x_t, u_t)
$$
観測：
$$
y_t = h(x_t)
$$
。

Observabilityとは：

観測 $y_t$ から
内部状態 $x_t$ を推定可能

なこと。




## 6. AHEの3種類のObservability

### 6.1 Component Observability

Harnessを：

```text id="u7k3pr"
file-level component
```

へ分解。

例：

```text id="n2c6dv"
workspace/
  middleware/
  tools/
```

。

これにより：

```text id="a6y1mh"
failure
→ 修正component
```

が対応する。

### 6.2 Experience Observability

巨大trajectoryを：

```text id="h5r9jw"
trajectory
↓
root-cause analysis
↓
failure pattern
```

へ圧縮。

例えば：

```text id="c3l0qe"
retry
cleanup
submit
```

というログを：

```text id="v9m4zk"
proxy validation instead of evaluator validation
```

へ抽象化する。

### 6.3 Decision Observability

各editに：

```json id="b2t7cn"
{
  "predicted_fixes": [...],
  "risk_tasks": [...]
}
```

を付与。

つまり編集を：

```text id="r1p8yb"
検証可能仮説
```

として扱う。



## 7. Middlewareの重要性

AHEでは middleware が極めて重要。

例：

```text id="j7q3lu"
ExecutionRiskHintsMiddleware
```

。

shell履歴から：

* shallow validation
* repeated retry
* localhost-only check

などを検出し、

次turnで：

```text id="d6v0rs"
FRAMEWORK reminder
```

を注入する。

これは：

```text id="x8z2af"
prompt engineering
↓
runtime engineering
```

への変化を意味する。



## 8. Publish-state Guard

AHEでは：

```text id="k9m5dt"
成功後cleanupして成果物を消す
```

問題が発生した。

対策として：

```text id="q4e7hy"
成功状態をruntimeで保護
```

した。

つまり：

```text id="m1b6uc"
LLMへのお願い
↓
execution-time enforcement
```

への移行である。



## 9. まとめ

Harnessとは：

> LLMを
> 「文章生成器」
> から
> 「実行可能Agent」
> に変える実行環境

である。

Observabilityとは：

> failure と edit効果を
> 構造的に観測可能にすること

である。

AHEは：

```text id="t7c4va"
Prompt Engineering
↓
Harness Engineering
↓
Observability-driven Evolution
```

への移行を示した重要研究である。



## 参考文献

[Lin2026AHE] Lin, J., Liu, S., Pan, C., Lin, L., Dou, S., Huang, X., Yan, H., Han, Z., & Gui, T. *Agentic Harness Engineering: Observability-Driven Automatic Evolution of Coding-Agent Harnesses*. arXiv:2604.25850 (2026).


---




エージェントが外部ツールを呼び出す仕組みは [MCP](8300.0130.MCP.ipynb) で扱います。